## **import**

In [11]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from groq import Groq
load_dotenv()

True

# **1-ingestion pipeline**

## **load documents**

In [2]:
docs_data = "docs"
if not  os.path.exists(docs_data):
        raise FileNotFoundError("the data file was not found")

loader = DirectoryLoader(
        path = docs_data,
        glob = "*.txt",
        loader_cls = TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )

documents = loader.load()
if len(documents) == 0:
    raise FileNotFoundError(f"no .txt files in {docs_data}")
for i,doc in enumerate(documents[:2]):
    print(f"document {i+1}  :")
    print(doc.metadata['source'])

document 1  :
docs\Google.txt
document 2  :
docs\Microsoft.txt


## **chunk documents**

In [3]:
chunk_size = 800
chunk_overlap = 0
text_splitter = CharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap
    )
chunks = text_splitter.split_documents(documents)
for i,chunk in enumerate(chunks[:2]):
    print(f"--- chunk {i+1} ---")
    print(f"source : {chunk.metadata['source']}")
    print(f"length : {len(chunk.page_content)} characters")
    print(chunk.page_content)

Created a chunk of size 949, which is longer than the specified 800
Created a chunk of size 922, which is longer than the specified 800
Created a chunk of size 892, which is longer than the specified 800
Created a chunk of size 825, which is longer than the specified 800
Created a chunk of size 921, which is longer than the specified 800
Created a chunk of size 830, which is longer than the specified 800
Created a chunk of size 1055, which is longer than the specified 800
Created a chunk of size 874, which is longer than the specified 800
Created a chunk of size 1436, which is longer than the specified 800
Created a chunk of size 924, which is longer than the specified 800
Created a chunk of size 815, which is longer than the specified 800
Created a chunk of size 1039, which is longer than the specified 800
Created a chunk of size 1078, which is longer than the specified 800
Created a chunk of size 1043, which is longer than the specified 800
Created a chunk of size 880, which is longe

--- chunk 1 ---
source : docs\Google.txt
length : 600 characters
﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and technology
company focusing on online advertising, search engine
technology, cloud computing, computer software,
quantum computing, e-commerce, consumer
electronics, and artificial intelligence (AI).[9] It has
been referred to as "the most powerful company in the The Google logo used since 2015
world" by the BBC[10] and is one of the world's most
valuable brands.[11][12][13] Google's parent company,
Alphabet Inc., is one of the five Big Tech companies
alongside Amazon, Apple, Meta, and Microsoft.
--- chunk 2 ---
source : docs\Google.txt
length : 738 characters
Google was founded on September 4, 1998, by
American computer scientists Larry Page and Sergey
Brin. Together, they own about 14% of its publicly
listed shares and control 56% of its stockholder voting
power through super-voting stock. The company went
public via an in

## **create vector db + filling it with chunks**

In [4]:
dir = "db/chroma_db"
embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5"
    )
print("--- creating vector database ---")
vector_store = Chroma(
        persist_directory=dir,
        embedding_function=embedding_model,
        collection_metadata={"hnsw:space":"cosine"}
    )

vector_store.add_documents(chunks)
print("--- finished creating vector database ---")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5006.76it/s]


--- creating vector database ---
--- finished creating vector database ---


# **2-retrieval**

In [18]:
query = "when did microsft joined the operating system business"

retriever = vector_store.as_retriever(search_kwargs={"k":3})

relevant_docs = retriever.invoke(query)

for i,rel_doc in enumerate(relevant_docs):
    print(f"--- document {i+1} ---")
    print(f"content : {rel_doc.page_content}")

--- document 1 ---
content : Microsoft entered the operating system (OS) business in 1980
with its own version of Unix called Xenix,[18] but it was MS-DOS that solidified the company's
dominance. IBM awarded a contract to Microsoft in November 1980 to provide a version of the CP/M
OS to be used in the IBM Personal Computer (IBM PC).[19] For this deal, Microsoft purchased a CP/M
clone called 86-DOS from Seattle Computer Products which it branded as MS-DOS, although IBM
rebranded it to IBM PC DOS. Microsoft retained ownership of MS-DOS following the release of the
IBM PC in August 1981. IBM had copyrighted the IBM PC BIOS, so other companies had to reverse
engineer it for non-IBM hardware to run as IBM PC compatibles, but no such restriction applied to the
operating systems. Microsoft eventually became the leading PC operating systems vendor.[20][21]: 210  The
company expanded into new markets with the release of the Microsoft Mouse in 1983, as well as with a
publishing division named Mi

# **3-LLM**

In [19]:
groq_api_key = os.getenv("GROQ_API_KEY")
Client = Groq(api_key=groq_api_key)

messages = [{
             "role":"system",
             "content":
             f"""
             here are some documents : 
            {chr(10).join([f"- {doc.page_content}" for doc in relevant_docs])}
            please provide an answer to the following question only based on the documents, if you dont have enought information in the 
            documents to answer, just say it
             """
           },
           {
               "role":"user",
               "content":f"question : {query}"
           }
    ]

response = Client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=messages,
                temperature=0
            )
content = response.choices[0].message.content
print(content)


Microsoft entered the operating system (OS) business in **1980**.
